# FastAPI Runner — Colab T4 + ngrok

Bu notebook, lokal makinedeki 4 GB VRAM kısıtı yüzünden Qwen2.5-VL-7B'yi çalıştıramayan setup için bir köprü kurar:

```
[Lokal PyCharm] → git push → [GitHub] → git pull → [Colab]
                                                      │
                                                      ├── uvicorn (FastAPI + Qwen + SBERT)
                                                      └── ngrok tunnel → public URL
                                                                              │
                                                                              ↓
                                                                      [Lokal Spring Boot]
                                                                      application.properties
                                                                      fastapi.base-url = <ngrok URL>
```

**Çalıştırmadan önce:**
1. Runtime → Change runtime type → **T4 GPU**
2. SBERT modelini Google Drive'a yüklemiş ol (talimat Bölüm 4'te)
3. https://dashboard.ngrok.com → AuthToken bölümünden token al (bedava, hesap aç)

## 1. GPU kontrolü

In [ ]:
!nvidia-smi

Beklenen: **NVIDIA Tesla T4, 16 GB VRAM**.
Eğer A100/V100 çıkarsa daha da iyi — Colab Pro'da bazen verir.

## 2. Repo'yu çek

GitHub'daki güncel kodu klonla. Public repo olduğu için auth gerek yok. Private ise PAT (Personal Access Token) gerekir.

In [ ]:
import os

REPO_URL = 'https://github.com/nebiavsar/tez_fastAPI.git'
REPO_DIR = '/content/tez_fastAPI'

if os.path.exists(REPO_DIR):
    # Daha önce klonlanmış — sadece güncelle
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!ls

## 3. Bağımlılıkları kur

Colab'da torch + CUDA zaten kurulu (cu128). `requirements.txt`'teki diğer paketleri ekliyoruz + Colab-specific olan `pyngrok` ve `nest-asyncio`.

`pillow<12` constraint'i kritik (Colab pre-installed torchvision Pillow 12.x ile uyumsuz).

In [ ]:
!pip install -q -r requirements.txt pyngrok nest-asyncio

In [ ]:
# Bağımlılık + versiyon doğrulama
import torch, transformers, bitsandbytes
print(f'torch         : {torch.__version__} (cuda={torch.cuda.is_available()})')
print(f'transformers  : {transformers.__version__}')
print(f'bitsandbytes  : {bitsandbytes.__version__}')
if torch.cuda.is_available():
    print(f'GPU           : {torch.cuda.get_device_name(0)}')
    print(f'VRAM          : {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB')

## 4. SBERT modelini Drive'dan kopyala

Model dosyası 449 MB → GitHub repo'sunda yok (.gitignore). Drive üzerinden taşıyacağız.

**İlk kullanımda** yapman gerekenler:
1. https://drive.google.com aç
2. Yeni klasör: **`tez_kaynaklari`** (My Drive altında)
3. Lokal makinendeki `C:\Users\nebi\Desktop\tez\tez_fastAPI\model_ai_noted_with_negatives_positives_v2` klasörünü bu klasöre **drag-drop ile yükle**
4. Yükleme bitince aşağıdaki hücreyi çalıştır

Sonraki çalıştırmalarda Drive zaten bağlı, kopyalama da idempotent (varsa atlar).

In [ ]:
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')

DRIVE_MODEL_DIR = '/content/drive/MyDrive/tez_kaynaklari/model_ai_noted_with_negatives_positives_v2'
LOCAL_MODEL_DIR = '/content/tez_fastAPI/model_ai_noted_with_negatives_positives_v2'

if not os.path.exists(DRIVE_MODEL_DIR):
    raise FileNotFoundError(
        f'Model klasörü Drive\'da bulunamadı: {DRIVE_MODEL_DIR}\n'
        f'4. bölümdeki talimatı takip ederek upload et.'
    )

if not os.path.exists(LOCAL_MODEL_DIR):
    print(f'Model kopyalanıyor: Drive → {LOCAL_MODEL_DIR}...')
    shutil.copytree(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR)
    print('Kopyalama tamam.')
else:
    print(f'Model zaten {LOCAL_MODEL_DIR} altında, atlanıyor.')

!ls -lh {LOCAL_MODEL_DIR}/model.safetensors

## 5. Uvicorn'u arka planda başlat

FastAPI lifespan event'i tetiklenince Qwen2.5-VL-7B (ilk seferde indirme ~6 GB) + SBERT yüklenir.

**İlk çalıştırma:** ~5-10 dakika (Qwen indirme).
**Sonraki çalıştırmalar:** ~60-90 saniye (cache'li).

In [ ]:
import subprocess
import time
from pathlib import Path

LOG_FILE = Path('/content/uvicorn.log')
if LOG_FILE.exists():
    LOG_FILE.unlink()

uvicorn_proc = subprocess.Popen(
    ['uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000', '--log-level', 'info'],
    cwd='/content/tez_fastAPI',
    stdout=open(LOG_FILE, 'w'),
    stderr=subprocess.STDOUT,
)
print(f'Uvicorn PID: {uvicorn_proc.pid}')
print('Modeller yükleniyor (ilk sefer ~5-10 dk Qwen indirme)...')

In [ ]:
# Yükleme ilerlemesini takip et. "sunucu hazır" mesajını bekliyoruz.
# İlk çalıştırmada bu hücre 5-10 dk sürer.
import time

ready = False
deadline = time.time() + 900  # 15 dakika max bekle
while time.time() < deadline:
    if LOG_FILE.exists():
        content = LOG_FILE.read_text(errors='ignore')
        if 'sunucu hazır' in content or 'Application startup complete' in content:
            ready = True
            print('Uvicorn hazır.')
            break
        if 'CUDA yok' in content or 'Error' in content:
            print('Hata var, log:')
            print(content[-2000:])
            break
    time.sleep(15)
    print(f'.. bekleniyor ({int(time.time() - (deadline - 900))}s)')

if not ready:
    print('15 dakika doldu, log son 2000 karakter:')
    print(LOG_FILE.read_text(errors='ignore')[-2000:])

In [ ]:
# Log'un son kısmını göster (VRAM bilgisi vs.)
!tail -30 /content/uvicorn.log

## 6. Ngrok tunnel

FastAPI'yi public internete açıyoruz ki lokal'deki Spring Boot çağırabilsin.

**İlk seferinde:** https://dashboard.ngrok.com/get-started/your-authtoken adresinden token al (bedava hesap).

In [ ]:
from pyngrok import ngrok, conf
from getpass import getpass

# Auth token'ı her runtime başlatmada bir kere ver — Colab değişkenleri RAM'de tutar.
if not conf.get_default().auth_token:
    token = getpass('ngrok auth token: ')
    ngrok.set_auth_token(token)

tunnel = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url
print(f'Public URL: {PUBLIC_URL}')
print(f'Swagger:    {PUBLIC_URL}/docs')
print()
print('Spring Boot application.properties:')
print(f'    fastapi.base-url={PUBLIC_URL}')

## 7. Smoke test — /health

In [ ]:
import requests

r = requests.get(f'{PUBLIC_URL}/health', timeout=10)
print(f'Status: {r.status_code}')
print(f'Body:   {r.json()}')

## 8. End-to-end test — /process-exam

Gerçek sınav fotoğrafı + örnek answer_key ile tam pipeline'ı çalıştır.

**Görüntü yükle:** aşağıdaki hücreyi çalıştırınca Choose Files çıkar, `ornek3.jpeg`'i seç.

In [ ]:
from google.colab import files

uploaded = files.upload()
image_filename = list(uploaded.keys())[0]
print(f'Yüklenen: {image_filename}')

In [ ]:
## 8. End-to-end test — /process-exam (v2 sözleşme)

Yeni sözleşme **iki görsel** ister:
- `paperImage`: öğrenci tarafından doldurulmuş sınav kâğıdı
- `answerKeyImage`: öğretmenin doğru cevap kâğıdı

Cevap kâğıdı OCR sonucu cache'lenir — aynı sınıf için sonraki öğrenci sınavlarında yeniden OCR yapılmaz.

**Pipeline testi için:** Eğer ayrı bir cevap kâğıdı fotoğrafın yoksa, `ornek3.jpeg`'i iki kez yükle (öğrenci + cevap kâğıdı). Bu durumda skor düşük olur (öğrenci ve referans aynı yanlış cevaplara sahip) — ama pipeline'ın çalıştığını görürsün.

**Gerçek skor için:** Hem öğrenci kâğıdı hem ayrı bir cevap kâğıdı fotoğrafı gerekli.

from google.colab import files

print('═════════════════════════════════════════════════')
print('1) ÖĞRENCİ kâğıdını yükle:')
print('═════════════════════════════════════════════════')
uploaded_paper = files.upload()
paper_filename = list(uploaded_paper.keys())[0]
print(f'   → {paper_filename}\n')

print('═════════════════════════════════════════════════')
print('2) ÖĞRETMEN CEVAP kâğıdını yükle:')
print('   (pipeline testi için aynı dosyayı tekrar seçebilirsin)')
print('═════════════════════════════════════════════════')
uploaded_ak = files.upload()
answer_key_filename = list(uploaded_ak.keys())[0]
print(f'   → {answer_key_filename}')

import json

# Yeni sözleşme: paperImage + answerKeyImage (iki görsel multipart)
# answer_key JSON artık kullanılmıyor — referans cevaplar cevap kâğıdı görselinden OCR edilir.

print('Çağrı gönderiliyor... (iki OCR + skorlama, ~3-5 dakika)')
with open(paper_filename, 'rb') as paper_f, open(answer_key_filename, 'rb') as ak_f:
    response = requests.post(
        f'{PUBLIC_URL}/process-exam',
        files={
            'paperImage': (paper_filename, paper_f, 'image/jpeg'),
            'answerKeyImage': (answer_key_filename, ak_f, 'image/jpeg'),
        },
        timeout=600,
    )

print(f'Status: {response.status_code}')

if response.status_code != 200:
    print('HATA:')
    print(json.dumps(response.json(), ensure_ascii=False, indent=2))
else:
    result = response.json()
    print(f'Toplam skor: {result.get("score")}')
    print()
    for q in result.get('questions', []):
        print(f'═══ Soru {q["questionId"]} (skor: {q["score"]}) ═══')
        print(f'  Öğrenci  : {q["extractedAnswer"]}')
        print(f'  Referans : {q["expectedAnswer"]}')
        print(f'  Geri bld : {q["feedback"]}')
        print()

In [ ]:
# Çalışmayı bitirmek için bu hücreyi çalıştır
ngrok.disconnect(tunnel.public_url)
ngrok.kill()
uvicorn_proc.terminate()
uvicorn_proc.wait(timeout=10)
print('Tunnel kapandı, uvicorn durduruldu.')